In [1]:
!pip install -q transformers datasets sacrebleu sentencepiece accelerate

In [2]:
from google.colab import files
files.upload()  # upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d mathurinache/flores101
!unzip -q flores101.zip -d flores101
!ls flores101

Saving kaggle.json to kaggle (1).json
Dataset URL: https://www.kaggle.com/datasets/mathurinache/flores101
License(s): CC-BY-NC-SA-4.0
flores101.zip: Skipping, found more recently modified local copy (use --force to force download)
replace flores101/flores101_dataset/README? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace flores101/flores101_dataset/dev/afr.dev? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace flores101/flores101_dataset/dev/amh.dev? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
flores101_dataset


In [3]:
import os

base_path = "/content/flores101/flores101_dataset/devtest"

def load_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [l.strip() for l in f]

SRC = "eng"
TGT = "zul"

sources = load_lines(f"{base_path}/{SRC}.devtest")
references = load_lines(f"{base_path}/{TGT}.devtest")

print("Eval samples:", len(sources))

Eval samples: 1012


In [4]:
from datasets import load_dataset

# Example: replace with your chosen corpus
dataset = load_dataset("opus100", "en-zu")

train_data = dataset["train"].select(range(20000))  # small subset for Colab

def preprocess(example):
    return {
        "src": example["translation"]["en"],
        "tgt": example["translation"]["zu"]
    }

train_data = train_data.map(preprocess)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "AfriNLP/AfriNLLB-12enc-12dec-full-ft-kd"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

config.json:   0%|          | 0.00/837 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [6]:
def tokenize(example):
    model_inputs = tokenizer(
        example["src"],
        text_target=example["tgt"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

    return model_inputs

tokenized_train = train_data.map(tokenize, batched=True)

print(tokenized_train[0].keys())

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

dict_keys(['translation', 'src', 'tgt', 'input_ids', 'attention_mask', 'labels'])


In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./afrinllb-retrained",
    per_device_train_batch_size=8,
    num_train_epochs=2,  # keep small for Colab
    learning_rate=2e-5,
    logging_steps=100,
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
model.save_pretrained("afrinllb-retrained")
tokenizer.save_pretrained("afrinllb-retrained")

In [ ]:
import time

def translate(texts, batch_size=8):
    tokenizer.src_lang = "eng_Latn"
    outputs = []

    start = time.time()

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids("zul_Latn"),
                max_length=200
            )

        decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
        outputs.extend(decoded)

    total = time.time() - start
    latency = total / len(texts)
    throughput = len(texts) / total

    return outputs, latency, throughput

In [ ]:
predictions, latency, throughput = translate(sources)

print("Latency:", latency)
print("Throughput:", throughput)

In [ ]:
import sacrebleu

chrf = sacrebleu.corpus_chrf(predictions, [references])
print("chrF++:", chrf.score)

In [ ]:
import json

# =========================================================
# Save retrained model metrics
# =========================================================

results = {
    "model": "AfriNLLB-retrained",
    "source_lang": "eng_Latn",
    "target_lang": "zul_Latn",
    "evaluation_dataset": "FLORES-original",
    "training_data": "OPUS100",
    "chrF++": chrf.score,
    "latency": latency,
    "throughput": throughput,
    "num_samples": len(sources)
}

# Save metrics
with open("retrained_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

# =========================================================
# Save retrained model predictions
# =========================================================

with open("retrained_predictions.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2)

# =========================================================
# Optional but strongly recommended:
# Save evaluation data for separate AfriCOMET notebook
# =========================================================

with open("retrained_sources.json", "w", encoding="utf-8") as f:
    json.dump(sources, f, indent=2)

with open("retrained_references.json", "w", encoding="utf-8") as f:
    json.dump(references, f, indent=2)

print("Saved retrained evaluation files.")